# Used Car Price Prediction

## 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

## 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [2]:
df = pd.read_csv("cardekho_imputated.csv", index_col=[0])

In [3]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Data Cleaning

### Handling Missing values


* Handling Missing values 
* Handling Duplicates
* Check data type
* Understand the dataset

In [6]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [7]:
## Remove Unnecessary Columns
df.drop('car_name', axis=1, inplace=True)
df.drop('brand', axis=1, inplace=True)

In [8]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [9]:
## Getting All Different Types OF Features
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical Features :', len(num_features))
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorical Features :', len(cat_features))
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features :',len(discrete_features))
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features :',len(continuous_features))

Num of Numerical Features : 7
Num of Categorical Features : 4
Num of Discrete Features : 2
Num of Continuous Features : 5


In [10]:
## Indpendent and dependent features
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

## Feature Encoding and Scaling
**One Hot Encoding for Columns which had lesser unique values and not ordinal**
* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [11]:
len(df['model'].unique())

120

This is a very huge number for OHE

In [12]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [13]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
X['model']=le.fit_transform(X['model'])

In [14]:
# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((12328, 10), (3083, 10))

In [15]:
# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
onehot_columns = ['seller_type','fuel_type','transmission_type']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, onehot_columns),
        ("StandardScaler", numeric_transformer, num_features)
        
    ],remainder='passthrough'
    
)

In [16]:
X_train=preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [18]:
pd.DataFrame(X_train)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.255968,0.323969,0.349100,-2.050819,1.756765,2.681685,-0.403824
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.789199,-1.337798,-1.069394,0.985661,-0.547081,-0.382744,-0.403824
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.242618,-1.337798,-1.163564,-0.177042,0.893542,3.296910,-0.403824
3,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.022962,0.323969,0.178369,-0.465315,0.024564,0.396229,-0.403824
4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.503081,1.321030,0.585469,0.149668,-0.550917,-0.502047,-0.403824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12323,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.556193,0.323969,1.702310,0.248161,-0.453086,-0.270460,2.070500
12324,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.899027,1.653383,0.084198,-0.876105,0.218310,0.066393,-0.403824
12325,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.036312,0.323969,-0.833967,0.185702,-0.932654,-0.779483,-0.403824
12326,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.530538,-1.337798,-0.951680,-0.273133,-0.550917,-0.432805,-0.403824


In [19]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [20]:
## Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [21]:
model = AdaBoostRegressor()
model.fit(X_train, y_train) # Train model

# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate Train and Test dataset
model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

print('Model performance for Training set')
print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
print("- R2 Score: {:.4f}".format(model_train_r2))

print('----------------------------------')

print('Model performance for Test set')
print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
print("- R2 Score: {:.4f}".format(model_test_r2))


Model performance for Training set
- Root Mean Squared Error: 467063.0746
- Mean Absolute Error: 352002.8267
- R2 Score: 0.7310
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 493104.1873
- Mean Absolute Error: 367501.3402
- R2 Score: 0.6770


In [23]:
ada_params={
    "n_estimators":[50,60,70,80],
    "loss":['linear','square','exponential']
}

In [25]:
## Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

random = RandomizedSearchCV(estimator=model,
                                param_distributions=ada_params,
                                n_iter=100,
                                cv=3,
                                verbose=2,
                                n_jobs=-1)
random.fit(X_train, y_train)

random.best_params_

Fitting 3 folds for each of 12 candidates, totalling 36 fits


{'n_estimators': 50, 'loss': 'linear'}

In [26]:
y_train_pred = random.predict(X_train)
y_test_pred = random.predict(X_test)

random_train_mae , random_train_rmse, random_train_r2 = evaluate_model(y_train, y_train_pred)

random_test_mae , random_test_rmse, random_test_r2 = evaluate_model(y_test, y_test_pred)

print('random performance for Training set')
print("- Root Mean Squared Error: {:.4f}".format(random_train_rmse))
print("- Mean Absolute Error: {:.4f}".format(random_train_mae))
print("- R2 Score: {:.4f}".format(random_train_r2))

print('----------------------------------')

print('random performance for Test set')
print("- Root Mean Squared Error: {:.4f}".format(random_test_rmse))
print("- Mean Absolute Error: {:.4f}".format(random_test_mae))
print("- R2 Score: {:.4f}".format(random_test_r2))


random performance for Training set
- Root Mean Squared Error: 422970.6984
- Mean Absolute Error: 298719.7059
- R2 Score: 0.7794
----------------------------------
random performance for Test set
- Root Mean Squared Error: 457435.2026
- Mean Absolute Error: 315397.8479
- R2 Score: 0.7220
